In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
from sklearn.metrics import f1_score, roc_auc_score
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
import pickle

import torch 
import random
from tqdm import tqdm

# Зафиксируем сиды, чтобы обучение было воспроизводимым.
seed = 1001
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(seed)

import warnings
warnings.filterwarnings("ignore")

# load data

In [2]:
user = pd.read_csv('users.csv')
user

,user_id,age,income,sex,kids_flg
0,523815,age_25_34,income_60_90,М,1
1,840727,age_18_24,income_20_40,М,0
2,558190,age_45_54,income_40_60,Ж,0
3,331576,age_45_54,income_20_40,Ж,0
4,749138,age_35_44,income_60_90,Ж,0
...,...,...,...,...,...
840192,159128,age_65_inf,income_0_20,Ж,0
840193,27937,age_18_24,income_20_40,Ж,1
840194,633813,NaN,NaN,NaN,0
840195,331859,NaN,NaN,Ж,0


In [3]:
item = pd.read_csv('items.csv')
item

,item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,keywords
0,10964,film,Item_000000,TitleOrig_000000,2004.0,"Genre_000000, Genre_000001, Genre_000002, Genr...",Country_000000,NaN,16.0,NaN,Director_000000,"Actor_000000, Actor_000001, Actor_000002, Acto...","Keyword_000000, Keyword_000001, Keyword_000002..."
1,2294,film,Item_000001,TitleOrig_000001,2016.0,"Genre_000001, Genre_000004, Genre_000005",Country_000001,NaN,16.0,NaN,Director_000001,"Actor_000020, Actor_000021, Actor_000022, Acto...","Keyword_000024, Keyword_000025, Keyword_000026..."
2,1691,film,Item_000002,TitleOrig_000002,2013.0,"Genre_000006, Genre_000001, Genre_000007, Genr...",Country_000002,NaN,16.0,NaN,Director_000002,"Actor_000040, Actor_000041, Actor_000042, Acto...","Keyword_000035, Keyword_000036, Keyword_000037..."
3,352,film,Item_000003,TitleOrig_000003,2017.0,"Genre_000000, Genre_000001, Genre_000003",Country_000003,NaN,16.0,NaN,Director_000003,"Actor_000054, Actor_000055, Actor_000056, Acto...","Keyword_000057, Keyword_000058, Keyword_000059..."
4,8281,film,Item_000004,NaN,1980.0,"Genre_000000, Genre_000009, Genre_000010, Genr...",Country_000004,NaN,12.0,Studio_000000,Director_000004,"Actor_000074, Actor_000075, Actor_000076, Acto...","Keyword_000072, Keyword_000073, Keyword_000074..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15958,13638,series,Item_015664,TitleOrig_010817,2020.0,"Genre_000000, Genre_000007, Genre_000006","Country_000050, Country_000007",NaN,16.0,NaN,Director_000585,"Actor_061870, Actor_061871, Actor_015894, Acto...","Keyword_000405, Keyword_006322, Keyword_012757..."
15959,10382,series,Item_001078,NaN,2022.0,"Genre_000000, Genre_000008",Country_000005,0.0,18.0,NaN,Director_001109,"Actor_003261, Actor_006717, Actor_007062, Acto...","Keyword_002708, Keyword_000280, Keyword_000088"
15960,1798,series,Item_015665,TitleOrig_010818,2019.0,"Genre_000000, Genre_000007, Genre_000006",Country_000005,0.0,18.0,NaN,"Director_008834, Director_008082, Director_008835","Actor_018960, Actor_014082, Actor_061876, Acto...","Keyword_043679, Keyword_000257, Keyword_000088"
15961,14241,series,Item_015666,TitleOrig_010819,2021.0,"Genre_000000, Genre_000009, Genre_000006",Country_000005,0.0,18.0,NaN,"Director_008836, Director_008743","Actor_061878, Actor_061879, Actor_061880, Acto...","Keyword_019329, Keyword_042747, Keyword_000246..."


# data analyze

In [4]:
user.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 840197 entries, 0 to 840196
Data columns (total 5 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   user_id   840197 non-null  int64 
 1   age       826102 non-null  object
 2   income    825421 non-null  object
 3   sex       826366 non-null  object
 4   kids_flg  840197 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 32.1+ MB


In [5]:
user_cat_cols = ['age', 'income', 'sex']

In [6]:
for col in tqdm(user_cat_cols):
    user[col].fillna(' ', inplace=True)

100%|██████████| 3/3 [00:00<00:00, 60.99it/s]


In [7]:
user.drop(columns = ['income', 'kids_flg', 'sex', 'age'], inplace = True)

In [8]:
user = pd.get_dummies(user)

In [9]:
item.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15963 entries, 0 to 15962
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   item_id       15963 non-null  int64  
 1   content_type  15963 non-null  object 
 2   title         15963 non-null  object 
 3   title_orig    11218 non-null  object 
 4   release_year  15865 non-null  float64
 5   genres        15963 non-null  object 
 6   countries     15926 non-null  object 
 7   for_kids      566 non-null    float64
 8   age_rating    15961 non-null  float64
 9   studios       1065 non-null   object 
 10  directors     14454 non-null  object 
 11  actors        13344 non-null  object 
 12  keywords      15540 non-null  object 
dtypes: float64(3), int64(1), object(9)
memory usage: 1.6+ MB


In [10]:
item.drop(columns = ['for_kids', 'studios', 'title_orig'], inplace = True)

In [11]:
item_cat_cols = ['content_type', 'title', 'genres', 'countries', 'directors', 'actors', 'keywords']
for col in tqdm(item_cat_cols):
    item[col].fillna(' ', inplace=True)

100%|██████████| 7/7 [00:00<00:00, 1092.14it/s]


# predict

In [ ]:
# test_df = pd.read_csv('interactions_public_test.csv')
test_df = pd.read_csv('interactions_private_test.csv')
test_df

,user_id,item_id,last_watch_dt,total_dur
0,17,846,2021-04-02,3288.0
1,25,11126,2021-07-31,9050.0
2,30,1372,2021-07-18,400.0
3,33,2865,2021-07-27,1507.0
4,33,9645,2021-08-06,732.0
...,...,...,...,...
100777,958957,14710,2021-07-18,212.0
100778,958966,10848,2021-06-19,11857.0
100779,958977,10848,2021-07-12,44675.0
100780,958983,7204,2021-07-15,65.0


In [ ]:
# sub = pd.read_csv('sample_sub_public_test_seed_0.csv')
sub = pd.read_csv('sample_sub_private_test_seed_0.csv')
sub

,user_id,item_id,last_watch_dt,total_dur,watched_pct
0,17,846,2021-04-02,3288.0,0.0
1,25,11126,2021-07-31,9050.0,0.0
2,30,1372,2021-07-18,400.0,0.0
3,33,2865,2021-07-27,1507.0,0.0
4,33,9645,2021-08-06,732.0,0.0
...,...,...,...,...,...
100777,958957,14710,2021-07-18,212.0,0.0
100778,958966,10848,2021-06-19,11857.0,0.0
100779,958977,10848,2021-07-12,44675.0,0.0
100780,958983,7204,2021-07-15,65.0,0.0


In [14]:
test_df['last_watch_dt'] = pd.to_datetime(test_df['last_watch_dt'])
test_df['watch_day'] = test_df['last_watch_dt'].dt.day
test_df['watch_month'] = test_df['last_watch_dt'].dt.month
test_df['watch_year'] = test_df['last_watch_dt'].dt.year

test_df.drop(columns = ['last_watch_dt'], inplace = True)

In [15]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100782 entries, 0 to 100781
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   user_id      100782 non-null  int64  
 1   item_id      100782 non-null  int64  
 2   total_dur    100782 non-null  float64
 3   watch_day    100782 non-null  int32  
 4   watch_month  100782 non-null  int32  
 5   watch_year   100782 non-null  int32  
dtypes: float64(1), int32(3), int64(2)
memory usage: 3.5 MB


In [16]:
test_df = test_df.merge(user, on = 'user_id', how = 'left')

In [17]:
test_df = test_df.merge(item, on = 'item_id', how = 'left')

In [18]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100782 entries, 0 to 100781
Data columns (total 15 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   user_id       100782 non-null  int64  
 1   item_id       100782 non-null  int64  
 2   total_dur     100782 non-null  float64
 3   watch_day     100782 non-null  int32  
 4   watch_month   100782 non-null  int32  
 5   watch_year    100782 non-null  int32  
 6   content_type  100782 non-null  object 
 7   title         100782 non-null  object 
 8   release_year  100778 non-null  float64
 9   genres        100782 non-null  object 
 10  countries     100782 non-null  object 
 11  age_rating    100782 non-null  float64
 12  directors     100782 non-null  object 
 13  actors        100782 non-null  object 
 14  keywords      100782 non-null  object 
dtypes: float64(3), int32(3), int64(2), object(7)
memory usage: 10.4+ MB


In [ ]:
# Load the model
model = SentenceTransformer("sparse-encoder-testing/splade-bert-tiny-nq")

test_dir_emb = []
for q in tqdm(test_df['directors']):
    test_dir_emb.append(model.encode(q))
np.save('test_dir_emb.npy', np.array(test_dir_emb))

test_act_emb = []
for q in tqdm(test_df['actors']):
    test_act_emb.append(model.encode(q))
np.save('test_act_emb.npy', np.array(test_act_emb))

test_key_emb = []
for q in tqdm(test_df['keywords']):
    test_key_emb.append(model.encode(q))
np.save('test_key_emb.npy', np.array(test_key_emb))

In [20]:
test_key_emb = np.load('test_key_emb.npy')
test_act_emb = np.load('test_act_emb.npy')
test_dir_emb = np.load('test_dir_emb.npy')

test_df['keywords'] = test_key_emb
test_df['actors'] = test_act_emb
test_df['directors'] = test_dir_emb

In [21]:
# load the model
filename = 'cat_model.sav'
model = pickle.load(open(filename, 'rb'))

In [22]:
pred = model.predict(test_df.drop(columns = ['user_id', 'item_id']))
pred

array([59.05273478, 98.89424607,  0.17535012, ..., 86.40826174,
        1.13139923, 12.60181505])

In [23]:
sub['watched_pct'] = pred
sub

,user_id,item_id,last_watch_dt,total_dur,watched_pct
0,17,846,2021-04-02,3288.0,59.052735
1,25,11126,2021-07-31,9050.0,98.894246
2,30,1372,2021-07-18,400.0,0.175350
3,33,2865,2021-07-27,1507.0,26.850133
4,33,9645,2021-08-06,732.0,12.765983
...,...,...,...,...,...
100777,958957,14710,2021-07-18,212.0,0.819229
100778,958966,10848,2021-06-19,11857.0,40.335945
100779,958977,10848,2021-07-12,44675.0,86.408262
100780,958983,7204,2021-07-15,65.0,1.131399


In [24]:
sub.to_csv(f'sub_{seed}.csv', index = False)